# Sentiment spot-check (Week 2)

**Owner:** Salah · **Checks:** Fatima's sentiment prototype · **Task:** manually spot-check about 20 scores for sanity

**Input needed from Fatima:** one CSV of scored text, saved to `data/processed/sentiment_prototype.csv`, with one row per scored text unit and these columns:

| Column | Meaning |
|---|---|
| `doc_id` | Filing ID (for example the EDGAR accession number) |
| `date` | Filing date |
| `section` | Where the text came from (for example `risk_factors`, `mda`) if known, else blank |
| `text` | The exact text the model scored (the chunk, not the whole filing) |
| `label` | Model label: `positive`, `negative` or `neutral` |
| `score` | Model confidence for that label, 0 to 1 |

**Steps:** run Part 1 to draw the sample and write the review sheet. Fill in `manual_label` and `notes` in the sheet by hand. Run Part 2 to score the check.

**Pass rule (proposed):** at least 14 of 20 manual labels agree with the model (70%), **and** no single pattern explains most of the misses. A systematic pattern is a fail even if agreement is high, because it will repeat across the full corpus in Week 3.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
SCORES = PROC / "sentiment_prototype.csv"
SHEET = PROC / "sentiment_spot_check.csv"
SEED = 2302          # fixed so the same sample is drawn every time
N_PER_LABEL = {"positive": 7, "negative": 7, "neutral": 6}
LABELS = ["positive", "negative", "neutral"]

## Part 1: draw the sample and write the review sheet

Stratified by model label so all three labels are checked, even if one is rare. Within each label, half the rows are the model's most confident calls and half are random, so the check covers both clear cases and ordinary ones.

In [2]:
scores = pd.read_csv(SCORES)
required = {"doc_id", "date", "text", "label", "score"}
missing_cols = required - set(scores.columns)
assert not missing_cols, f"sentiment_prototype.csv is missing columns: {missing_cols}"
scores["label"] = scores["label"].str.lower().str.strip()
if "section" not in scores:
    scores["section"] = ""

print(f"{len(scores)} scored rows from {scores['doc_id'].nunique()} documents")
print(scores["label"].value_counts().to_string())
print(f"\nText length (characters): median {scores['text'].str.len().median():.0f}, "
      f"max {scores['text'].str.len().max():.0f}")

parts = []
for lab, n in N_PER_LABEL.items():
    pool = scores[scores["label"] == lab]
    if len(pool) <= n:
        parts.append(pool.assign(pick="all available"))
        continue
    top = pool.nlargest(n // 2, "score").assign(pick="most confident")
    rest = pool.drop(top.index).sample(n - len(top), random_state=SEED).assign(pick="random")
    parts.append(pd.concat([top, rest]))
sample = pd.concat(parts).reset_index(drop=True)
sample.insert(0, "check_id", range(1, len(sample) + 1))
sample["manual_label"] = ""
sample["notes"] = ""

if SHEET.exists():
    print(f"\n{SHEET.name} already exists; not overwriting your manual labels.")
else:
    sample[["check_id", "doc_id", "date", "section", "pick", "label", "score", "manual_label", "notes", "text"]] \
        .to_csv(SHEET, index=False)
    print(f"\nWrote {len(sample)} rows to {SHEET.relative_to(ROOT)}")

50 scored rows from 19 documents
label
negative    23
neutral     21
positive     6

Text length (characters): median 386, max 1988

Wrote 19 rows to data\processed\sentiment_spot_check.csv


### How to label

Read each `text` and write `positive`, `negative` or `neutral` in `manual_label` **before** looking at the model's `label`, so the model does not anchor you. Judge the text as an investor would: does it signal better or worse prospects for the company?

In `notes`, record which of these applies when you disagree (these are the known weak spots of financial sentiment models on filings):

* `boilerplate`: standard risk-factor or legal language ("could adversely affect") scored negative even though it says nothing new
* `forward-looking`: the safe-harbor disclaimer about forward-looking statements
* `truncated`: the text was cut off; BERT-based models such as FinBERT read at most 512 tokens
* `numbers`: tables or figures the model cannot interpret
* `negation`: meaning flipped by "not", "no longer", "despite"
* `mixed`: good and bad news in the same chunk

## Part 2: score the check (run after filling in the sheet)

In [3]:
sheet = pd.read_csv(SHEET, dtype={"manual_label": str, "notes": str}).fillna({"manual_label": "", "notes": ""})
sheet["manual_label"] = sheet["manual_label"].str.lower().str.strip()
done = sheet[sheet["manual_label"].isin(LABELS)]
print(f"{len(done)} of {len(sheet)} rows labeled")

if len(done):
    agree = (done["manual_label"] == done["label"])
    n_ok, n = int(agree.sum()), len(done)
    print(f"Agreement: {n_ok} of {n} ({n_ok / n:.0%})")
    print("\nModel label (rows) vs manual label (columns):")
    display(pd.crosstab(done["label"], done["manual_label"]).reindex(index=LABELS, columns=LABELS, fill_value=0))
    print("\nAgreement by how the row was picked:")
    display(done.assign(agree=agree).groupby("pick")["agree"].agg(["sum", "count"]))
    misses = done[~agree]
    print("\nPatterns among disagreements (from notes):")
    tags = misses["notes"].str.lower().str.extractall(
        r"(boilerplate|forward-looking|truncated|numbers|negation|mixed)")[0]
    print(tags.value_counts().to_string() if len(tags) else "none tagged")
    top_share = tags.value_counts().iloc[0] / len(misses) if len(tags) else 0
    passed = (n_ok / n >= 0.70) and (top_share <= 0.5)
    print(f"\nResult: {'PASS' if passed else 'FAIL'} "
          f"(needs >= 70% agreement and no single pattern behind more than half the misses)")

19 of 19 rows labeled
Agreement: 12 of 19 (63%)

Model label (rows) vs manual label (columns):


manual_label,positive,negative,neutral
label,,,
positive,5,0,1
negative,0,2,5
neutral,0,1,5



Agreement by how the row was picked:


,sum,count
pick,,
all available,5,6
most confident,4,6
random,3,7



Patterns among disagreements (from notes):
0
boilerplate        4
numbers            1
truncated          1
forward-looking    1

Result: FAIL (needs >= 70% agreement and no single pattern behind more than half the misses)


In [4]:
# Disagreements, for the write-up to Fatima
pd.set_option("display.max_colwidth", 300)
done[done["manual_label"] != done["label"]][["check_id", "doc_id", "section", "label", "score", "manual_label", "notes", "text"]]

,check_id,doc_id,section,label,score,manual_label,notes,text
5,6,NVDA_10-K_2022-03-18,Item 7,positive,0.9061,neutral,numbers: expense increase read as good news,"Research and development expenses increased by 34% in fiscal year 2022 compared to fiscal year 2021, primarily driven by stock-based compensation, compensation-related costs associated with employee growth and higher infrastructure costs."
6,7,NVDA_10-K_2022-03-18,Item 1A,negative,0.9605,neutral,boilerplate,Any one or more of the factors discussed above could prevent us from achieving our expected future financial results. Any such failure to meet our expectations or the expectations of our investors or security analysts could cause our stock price to decline or experience substantial price volatil...
8,9,MSFT_10-Q_2022-10-25,Item 1A,negative,0.9555,neutral,boilerplate,"We may experience quality or supply problems. Our hardware products such as Xbox consoles, Surface devices, and other devices we design and market are highly complex and can have defects in design, manufacture, or associated software. We could incur significant expenses, lost revenue, and reputa..."
9,10,AMZN_10-Q_2022-10-28,Item 1A,negative,0.6193,neutral,truncated: list fragment,"• variability in revenue and income from entering into, amending, or terminating such agreements or relationships; and"
10,11,NVDA_10-Q_2022-05-27,Item 1A,negative,0.8839,neutral,boilerplate,"Before you buy our common stock, you should know that making such an investment involves some risks including, but not limited to, the risks described in Item 1A of our Annual Report on Form 10-K for the fiscal year ended January 30, 2022 and below. Additionally, any one of those risks could har..."
12,13,MSFT_10-Q_2022-01-25,Item 1A,negative,0.8452,neutral,boilerplate,"Third parties may claim we infringe their intellectual property rights. From time to time, others claim we infringe their intellectual property rights. The number of these claims may grow because of constant technological change in the markets in which we compete, the extensive patent coverage o..."
17,18,NVDA_10-Q_2022-08-31,Item 2,neutral,0.8085,negative,"forward-looking: hedged ""may"" language hides real news (China export license)","The new license requirement may impact our ability to complete our development of H100 in a timely manner or support existing customers of A100 and may require us to transition certain operations out of China, which could be costly and time consuming, and adversely affect our research and develo..."
